<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/review_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

Update to the similarity package installation the original code segmentation logic has used

In [ ]:
# Install the strsimpy (instead of similarity) research library
!pip install strsimpy -q

# Apply the "Bridge" Fixes
import sys
import strsimpy

# Alias strsimpy so the author's 'import similarity' works
sys.modules['similarity'] = strsimpy

In [23]:
# @title Dependencies

# Installation


# First-party installations
import os
from google.colab import drive
from IPython.display import display

# Third-party installations
import pandas as pd
import numpy as np
from tqdm import tqdm
import re
from strsimpy.normalized_levenshtein import NormalizedLevenshtein

In [24]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_segmented.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_segmented.jsonl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews.


In [25]:
# @title Data import

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(DATASET_DIR, "llm_reviews_results.parquet"))

# Select target columns
paper_content = dataset_paper.copy()

# Check paper_content
display(paper_content.head(3))
display(paper_content.shape)

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes_1,llm_review_1,llm_notes_2,llm_review_2
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...
1,author_3,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: Generating Sequences by Learning to [Se...,# Summary Of The Paper\nThis paper proposes Se...,"Paper: ""Generating Sequences by Learning to [S...",# Summary Of The Paper\nThe paper proposes SEL...
2,author_4,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,"Paper: ""A Statistical Framework for Personaliz...",# Summary Of The Paper\nThe paper proposes a u...,"Paper: ""A Statistical Framework for Personaliz...",# Summary Of The Paper\nThis paper proposes a ...


(16, 11)

## Utils

Helper functions from the authors utils.py which are prequists for the segmentation code logic: https://github.com/bodasadallah/RevUtil/blob/main/utils.py

In [26]:
def merge_short_sentences(paragpraphs):
    ret = []
    res = ''
    for p in paragpraphs:
        # if this paragraph is short, then add it to the next one
        if len(p.split()) < 5:
            res = res + p
        else:
            ret.append(res + p)
            res = ''
    if res:
        ret.append(res)

    return ret

# Filters reviews points  based on the length of the points.
# We only take revies that has a length of one STD away from the mean
def filter_reviews(df, review_field, exclude_short = True, exclude_long = False):

    print('filtering reviwes, and only considering the ones with length of one STD away form mean.')
    print('Number of reviews before filtering:', len(df))


    lengths = []
    num_points = 0
    for x in df[review_field].tolist():
        points  = x
        num_points += len(points)
        for point in points:
            lengths.append(len(point.split()))

    print('Number of the review points before filtering:', num_points)
    lengths = np.array(lengths)
    mean, std = np.mean(lengths.astype(int)), np.std(lengths.astype(int))

    min_length, max_length = mean- 1*std, mean+ 1*std
    print('mean:', mean, 'std:', std, 'min:', min_length, 'max:', max_length)


    num_points_after = 0
    for i,r in tqdm(df.iterrows(),total=len(df)):
        splitted_review = []
        for review in r[review_field]:
            if exclude_short and len(review.split()) < min_length:
                continue
            if exclude_long and len(review.split()) > max_length:
                continue

            splitted_review.append(review)

        num_points_after += len(splitted_review)
        df.at[i, 'split_review'] = splitted_review
    print('Number of the reviews after filtering:', num_points_after)

    return df

def clean_text(text):

    # Remove these words if they occur in the beginning of the text "Weaknesses, Strengths, Comments, Suggestions, Feedback" these can be followed by : , and can be lowe case
    text = re.sub(r'^(weaknesses|strengths|comments|suggestions|feedback)[:]*', '', text, flags=re.IGNORECASE)


    # Replace multiple spaces with a single space
    text = re.sub(r'[ ]{2,}', ' ', text)
    # Replace multiple newlines with a single newline
    text = re.sub(r'\n{2,}', '\n', text)
    # Strip leading and trailing spaces/newlines from each line
    text = '\n'.join(line.strip() for line in text.splitlines())
    text = text.replace('  ',' ')


    lines = text.split('\n')
    new_lines = []
    for l in lines:
        if l.strip():
            if len(l.split()) < 2 and new_lines:
                new_lines[-1] = new_lines[-1] + ' ' + l
            else:
                new_lines.append(l)

    text = '\n'.join(new_lines)

    return text

## Segmentation logic

Original segmentation code from: https://github.com/bodasadallah/RevUtil/blob/main/data_processing/rule_based_review_split.py

In [29]:
import os
import sys
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import re
import importlib
from similarity.normalized_levenshtein import NormalizedLevenshtein
module_path = Path(os.path.abspath("")).parent
print(module_path)
sys.path.append(str(module_path))


import re
import pandas as pd
#from utils import filter_reviews, merge_short_sentences, clean_text ### NO IMPORT NEEDED AS ABOVE CODE BLOCK INCLUDES THE NESSECARY FUNCTIONS
from tqdm import tqdm


normalized_levenshtein = NormalizedLevenshtein()


############ This has a limitation for the cases when the point enumeration becomes more than 9
############ This also have an issue with examples like "3.2.1" as it will be considered as a point
def split_into_points(text):
    splits_positions = []
    open_parantheses = 0
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    for i, char in enumerate(text):

        # keep track for the open parantheses, for the cases when we have (123)
        if char == '(':
            open_parantheses += 1
        if char == ')':
            open_parantheses -= 1

        ## Ideally, we don't want to split the text if we are inside a parantheses
        if open_parantheses > 0:
            continue

        # match points that starts with:  - , * , = followed by a space
        if char in ['-','*', '='] and text[i+1] == ' ':
            if i > 0:
                # make sure this point is preceded by a sentence ending or new line
                if  text[i-1] in ['.','!','?','\n'] or text[i-2] in ['.','!','?','\n']:
                    splits_positions.append(i)
            else:
                splits_positions.append(i)

        # If the char is a number
        if char.isnumeric():

            ## check for x.x and x.x.x:
            if text[i+1] == '.' and text[i+2].isnumeric():
                i = i+2
                continue


            ## match if the number is followed by: ')' or '.' or ':'
            if text[i+1] in [')','.',':'] and text[i+2] != '\n':
                if i > 1:
                    # make sure this point is preceded by a sentence ending or new line
                    if  text[i-1] in [' ','\n'] or text[i-2] in ['.','!','?','\n']:
                        splits_positions.append(i)
                else:
                    splits_positions.append(i)
            ## Match when we have a number at the begining of a new line
            elif (i == 0 or text[i-1] == '\n') and text[i+1] == ' ':
                splits_positions.append(i)

    splits_positions = [0] + splits_positions
    points = [ text[splits_positions[i]:splits_positions[i+1]].strip() for i in range(len(splits_positions)-1)]
    points.append(text[splits_positions[-1]:].strip())

    points = [point for point in points if point]
    return points


# Get the abimportsolute postion of each begining of a word in the text
def word_begging_postition(text):
    positions = []
    length = 0
    for word in text.split():
        positions.append(length)
        length += len(word)+1
    return positions


CONFERENCES = ['AAAI', 'AAMAS', 'ACL', 'ACMMM', 'ASPLOS', 'CAV', 'CCS', 'CHI', 'COLT', 'CRYPTO',
 'CSCL', 'DCC', 'DSN', 'FOCS', 'FOGA', 'HPCA', 'ICAPS', 'ICCV', 'ICDE', 'ICDM',
 'ICFP', 'ICIS', 'ICML', 'ICSE', 'IJCAI', 'IJCAR', 'INFOCOM', 'IPSN', 'ISCA',
 'ISMAR', 'ISSAC', 'ISWC', 'JCDL', 'KR', 'LICS', 'MOBICOM', 'NIPS', 'OOPSLA',
 'OSDI', 'PERCOM', 'PERVASIVE', 'PLDI', 'PODC', 'PODS', 'POPL', 'RSS', 'RTSS',
 'SENSYS', 'SIGCOMM', 'SIGGRAPH', 'SIGIR', 'SIGKDD', 'SIGMETRICS', 'SIGMOD',
 'SODA', 'SOSP', 'STOC', 'UAI', 'VLDB', 'WWW', 'ACM-HT', 'AH', 'AID', 'AIED',
 'AIIM', 'AIME', 'ALENEX', 'ALIFE', 'AMAI', 'AMIA', 'AOSD', 'APPROX', 'ASAP',
 'ASE', 'ASIACRYPT', 'ATVA', 'AVSS', 'BMVC', 'BPM', 'CADE', 'CAIP', 'CANIM',
 'CASES', 'CBSE', 'CC', 'CCC', 'CCGRID', 'CDC', 'CGI', 'CGO', 'CIDR', 'CIKM',
 'CLUSTER', 'COCOON', 'COLING', 'CONCUR', 'CP', 'CPAIOR', 'CSB', 'CSCW', 'CSFW',
 'CSSAC', 'CVPR', 'DAC', 'DAS', 'DASFAA', 'DATE', 'DEXA', 'DIGRA', 'DIS', 'DISC',
 'DOOD', 'DUX', 'EAAI', 'EACL', 'EASE', 'EC', 'ECAI', 'ECCV', 'ECDL', 'ECIS',
 'ECML', 'ECOOP', 'ECRTS', 'ECSCW', 'EDBT', 'EKAW', 'EMMSAD', 'EMNLP', 'EMSOFT',
 'ESA', 'ESEM', 'ESOP', 'ESORICS', 'ESQARU', 'ESWC', 'EUROGRAPH', 'EWSN', 'FCCM',
 'FLOPS', 'FME', 'FODO', 'FORTE', 'FPSAC', 'FSE', 'FSR', 'FUZZ-IEEE', 'GD',
 'HOTNETS', 'HPDC', 'ICADL', 'ICALP', 'ICALT', 'ICARCV', 'ICC', 'ICCAD', 'ICCL',
 'ICCS', 'ICCS', 'ICDAR', 'ICDCS', 'ICDT', 'ICECCS', 'ICER', 'ICGG', 'ICIAP',
 'ICIP', 'ICLP', 'ICMAS', 'ICNN', 'ICNP', 'ICONIP', 'ICPP', 'ICPR', 'ICS',
 'ICSM', 'ICSOC', 'ICSP', 'ICSPC', 'ICSR', 'ICTL', 'IDA', 'IEEE-CEC', 'IEEE-MM',
 'IEEETKDE', 'IJCNLP', 'IJCNN', 'ILPS', 'IM', 'IMC', 'INTERACT', 'IPCO', 'IPDPS',
 'ISAAC', 'ISD', 'ISESE', 'ISMB', 'ISR', 'ISSCC', 'ISSR', 'ISSRE', 'ISSTA',
 'ISTA', 'ISTCS', 'ISWC', 'ITS', 'IUI', 'IVCNZ', 'JELIA', 'K-CAP', 'LCN',
 'LCTES', 'LPAR', 'LPNMR', 'MASCOTS', 'MASS', 'MICRO', 'Middleware', 'MIR',
 'MMCN', 'MMSP', 'MOBIHOC', 'MobileHCI', 'Mobiquitous', 'Mobisys', 'MODELS',
 'MSWIM', 'NAACL', 'NDSS', 'NetStore', 'Networking 200X', 'NOSSDAV', 'NSDI',
 'OPENARCH', 'P2P', 'PACT', 'PADL', 'PADS', 'PAKDD', 'PDC', 'PEPM', 'PERFORMANCE',
 'PG', 'PKDD', 'PPoPP', 'PPSN', 'PRO-VE', 'PT', 'QoSA', 'QSIC', 'RAID', 'RANDOM',
 'RE', 'RECOMB', 'RoboCup', 'RST', 'RTA', 'RTAS', 'SARA', 'SAS', 'SAT', 'SCA',
 'SCC', 'SCG', 'SCOPES', 'SDM', 'SIGCSE', 'SMS', 'SPAA', 'SPICE', 'SRDS',
 'SSDBM', 'SSPR', 'SSR', 'SSTD', 'STACS', 'SUPER', 'SWAT', 'TABLEAUX', 'TACAS',
 'TARK', 'TIME', 'TREC', 'UIST', 'UM', 'USENIX', 'VIS', 'VL/HCC', 'VLSI',
 'VMCAI', 'WACV', 'WADS', 'WISE', 'WoWMoM', 'WPHOL', 'AAAAECC', 'AAIM', 'ACAL',
 'ACCV', 'ACE', 'ACIS', 'ACISP', 'ACIVS', 'ACOSM', 'ACRA', 'ACS', 'ACSAC',
 'ACSC', 'ACSD', 'ADBIS', 'ADC', 'ADCS', 'ADHOC-NOW', 'ADTI', 'AI*IA', 'AINA',
 'AISP', 'ALEX', 'ALG', 'ALP', 'ALTAW', 'AMCIS', 'AMOC', 'ANALCO', 'ANNIE',
 'ANTS', 'ANZIIS', 'AofA', 'AOIR', 'AOIS', 'AOSE', 'APAMI', 'APBC', 'APCC',
 'APCHI', 'APLAS', 'APNOMS', 'APSEC', 'APWEB', 'ARA', 'ARES', 'ASADM', 'ASIAN',
 'ASS', 'ASWEC', 'AUIC', 'AusAI', 'AusDM', 'AusWIT', 'AWOCA', 'AWRE', 'AWTI',
 'BASYS', 'BNCOD', 'Broadnets', 'CAAI', 'CAAN', 'CACSD', 'CAIA', 'CATS',
 'CCA', 'CCCG', 'CCW', 'CD', 'CEAS', 'CEC/EEE', 'CGA', 'CHES', 'CIAA', 'CIAC',
 'CICLING', 'CISTM', 'CITB', 'COCOA', 'COMAD', 'COMMONSENSE', 'CompLife',
 'COMPSAC', 'CONPAR', 'CPM', 'CSL', 'DAC', 'DAFX', 'DAIS', 'DaWaK', 'DB&IS',
 'DCOSS', 'DICTA', 'DISRA', 'DITW', 'DLT', 'DMTCS', 'DNA', 'DSOM', 'DS-RT',
 'DSS', 'DX', 'DYSPAN', 'ECAIM', 'ECAL', 'ECBS', 'ECCB', 'ECEG', 'ECIME',
 'ECIR', 'ED-MEDIA', 'EDOC', 'EEE', 'EGC', 'Emnets', 'EPIA', 'ER', 'ERCIM/CSCLPERCIM',
 'ESEA', 'ESEC', 'ESM', 'ESS', 'EuAda', 'EUROGP', 'EuroPDP', 'EUSIPCO',
 'EWLR', 'FASE', 'FCKAML', 'FCT', 'FEM', 'FEWFDB', 'FIE', 'FINCRY', 'FOSSACS',
 'FSENCRY', 'FTP', 'FTRTFT', 'FUN', 'GECCO', 'GLOBECOM', 'GMP', 'GPCE',
 'HASE', 'HICSS', 'HLT', 'HPCN', 'HPSR', 'IAAI', 'ICA3PP', 'ICAIL']

SKIP_WORDS = ['tab','fig', 'table', 'figure', 'sec', 'section','al', 'eqn', 'equation' , 'figs', 'eq', 'vol', 'volume', 'chap', 'chapter']
POINT_DELIMITERS = ['.', '-', '*', '•','_']
SENTENCE_ENDINGS = ['.','!','?', ':',';']

def split_into_points_new(text):
    splits_positions = []
    open_parantheses = 0
    # Remove extra spaces
    # text = re.sub(r'\s+', ' ', text)

    text = clean_text(text)

    ## remove post-rebuttal text
    post_rebuttal_phrases = ['post-rebuttal', 'post rebuttal', 'postrebuttal','After reading the response']
    for phrase in post_rebuttal_phrases:
        if phrase in text:
            text = text.split(phrase)[0]


    word_beging_possitions = word_begging_postition(text)
    new_line_positions = [i for i, char in enumerate(text) if char == '\n']
    words = text.split()
    words = [words.lower() for words in words]
    for i, word in enumerate(words):


        ## Custom case to split for General Discission

        if  (i < len(words) -1 ) and ('general' in word)   and ('discussion' in words[i+1]):
            splits_positions.append(word_beging_possitions[i])
            continue

        if open_parantheses <= 0:
            if word in POINT_DELIMITERS:
                if i > 0:
                    # make sure this point is preceded by a sentence ending or new line
                    word_begin_pos = word_beging_possitions[i]
                    # check if the word is at the begining of a new line or the last words ended with a sentence ending.
                    if  ( (word_begin_pos -1) in new_line_positions  or  (word_begin_pos -1) == '\n') or  ( words[i-1][-1] in SENTENCE_ENDINGS ):
                        splits_positions.append(word_beging_possitions[i])
                        continue
                else:
                    splits_positions.append(word_beging_possitions[i])
                    continue

            ## match (W123) or (w123)
            if re.match(r'\([w|W]\d+\)',word):
                splits_positions.append(word_beging_possitions[i])
                continue


            ## Match if this is a number, and not preceeded by a word that is a section or figure or table
            ## this matches the cases when we have NUM,  that is not a section or figure or table
            # This also make sure that we are not in the middle of a sentence
            if word.isnumeric():
                if i and words[i-1].replace('.','').replace(':','') \
                    not in (SKIP_WORDS + CONFERENCES)\
                    and words[i-1][-1] in SENTENCE_ENDINGS:

                    splits_positions.append(word_beging_possitions[i])
                    continue
                elif not i:
                    splits_positions.append(word_beging_possitions[i])
                    continue




            # This matches the cases when we have NUM that is followed by a '.' or ':' or ')'
            # This also make sure that we are not in the middle of a sentence
            # make sure this is not a NUM.NUM
            if re.match(r"[a-zA-z]*\d+[.:)]", word) and not re.match(r"\d+.\d+", word):

                if i and (words[i-1][-1] in SENTENCE_ENDINGS):
                    splits_positions.append(word_beging_possitions[i])
                elif not i:
                    splits_positions.append(word_beging_possitions[i])

                ### special case if we have NUM) then we don't need to be in the begin of a sentence
                elif ')' in word:
                    splits_positions.append(word_beging_possitions[i])

                # if this has a closed parantheses, then we add a dummy ones, so we don't mess the count
                if ')' in word:
                    open_parantheses += word.count(')')



        # keep track for the parantheses
        open_parantheses += word.count('(')
        open_parantheses -= word.count(')')


    splits_positions = [0] + splits_positions
    points = [ text[splits_positions[i]:splits_positions[i+1]].strip() for i in range(len(splits_positions)-1)]
    points.append(text[splits_positions[-1]:].strip())

    points = [point for point in points if point]
    return points



ARROWS= ['->', '=>','-->','==>', '→', '⟶']
## Filters review points that are typos fixes, and review points that doesn't start with dot, dash, star, or number
def filter_typos(points):
    quoted = r'["\'“”](.*?)["\'“”]'
    for point in list(points):

        # remove the point if it has the word typo
        if 'typo' in point.lower():
            points.remove(point)
            continue

        erased = False
        # Filter out the points that are typos fixes. We detect typos with the following patterns:
        for x in ARROWS:
            if x in point.split():
                qouted_text = re.findall(quoted,point)
                # print(qouted_text)
                for i, q in enumerate(qouted_text):
                    for j, q2 in enumerate(qouted_text):
                        if i != j  and normalized_levenshtein.distance(q, q2) < 0.6:
                            points.remove(point)
                            erased = True
                            break
                    if erased:
                        break
            if erased:
                break


    return points

def only_bullet_points(points):
    filtered_points = []
    for point in list(points):
        # Filter out the points that doesn't start with dot, dash, star, or number
        if point[0] in POINT_DELIMITERS or point[0].isnumeric():
            filtered_points.append(point)
    return filtered_points



################### Still unfinished ##################
def split_with_llm(points, client):
    points = []

    prompt = '''The following is a review of a scientific paper. Read it carefully, and decide if there are multiple coherent points in this review, and if they can be split into more than one point.  Don't change anything in the text. write the text as is. Output the different points in the format:
POINT#: [point]
Review:
{review_point}
'''

    for point in points:

        current_point = prompt.format(review_point = point)
        completion = client.chat.completions.create(
        messages=[
        {"role": "user", "content": current_point}],)
        response = completion.choices[0].message.content.lower().strip()

        pattern = r'POINT\d+'
        matches = re.findall(pattern, response)



def remove_post_rebuttal(points):
    remove_list = ['------','******','======','______', 'rebuttal']
    clean_points = []
    for point in points:
        if any([x in point for x in remove_list]):
            continue
        clean_points.append(point)
    return clean_points


def split_and_filter(df,
                    review_key,
                    filter_short_reviews=True,
                    do_filter_typos=True,
                    consider_only_bullet_points=True,
                    exclude_short = False,
                    exclude_long = False,
                    split_with_llm = False,
                    llm_client = None):

    all_cnt = 0
    cnt_after_typos_bullet_points = 0
    cnt_one_point_reviews = 0

    split_review_column = []

    print(f'Initial number of reviews: {len(df)}')
    ## Filtering reviews to remove ones with no valid lists
    df = df[df[review_key].apply(lambda x: len(x) > 0)]
    df = df.dropna(subset=[review_key])
    print('Number of reviews after removing  zero-length review', len(df))


    ## Split the reviews into points
    for i,r in tqdm(df.iterrows(),total=len(df)):
        focused_review = re.sub(r'\s+', ' ', r[review_key])
        ## remove the first occurence of weakness or Weakness
        remove_list = ['weakness', 'Weakness','weaknesses', 'Weaknesses']
        for word in remove_list:
            if focused_review.startswith(word) or focused_review.startswith(word + ':'):
                focused_review  = ' '.join(focused_review.split()[1:])

        # Split thre review, and merge short sentences
        splitted_reviews = split_into_points_new(focused_review)
        splitted_reviews = merge_short_sentences(splitted_reviews)

        if split_with_llm:
            splitted_reviews = split_with_llm(splitted_reviews, llm_client)


        ####################### Removing reviews with one point #######################
        if len(splitted_reviews) == 1:
            cnt_one_point_reviews += 1
            splitted_reviews = []


        all_cnt += len(splitted_reviews)


        ## remove reviews that are post rebuttal

        if filter_short_reviews:
        ########## Filter short reviews  ###################
            for review in list(splitted_reviews):
                if len(review.split()) < 10:
                    splitted_reviews.remove(review)


        if do_filter_typos:
            splitted_reviews = filter_typos(splitted_reviews)

        if consider_only_bullet_points:
            splitted_reviews = only_bullet_points(splitted_reviews)

        cnt_after_typos_bullet_points += len(splitted_reviews)


        ############ Remove reviews with post-rebuttal text ################
        splitted_reviews =  remove_post_rebuttal(splitted_reviews)



        split_review_column.append(splitted_reviews)


    df['split_review'] = split_review_column


    print('Number of reviews with one point: (these reviews will be excluded)', cnt_one_point_reviews)
    print(f'Number of all points (for reviews that has more than one point): {all_cnt}')
    print(f'Number of points after filtering typos and considering only bullet points: {cnt_after_typos_bullet_points}')

    #drop Empty reviews
    df.drop(df[df['split_review'].apply(lambda x: len(x) == 0)].index, inplace=True)

    ### Filter short reviews
    df = filter_reviews(df, 'split_review', exclude_short = exclude_short, exclude_long = exclude_long)

    #drop Empty reviews
    df.drop(df[df['split_review'].apply(lambda x: len(x) == 0)].index, inplace=True)


    return df

# Example usage:
if __name__ == "__main__":
    print('Running the example usage')

/
Running the example usage


## Execution

In [30]:
# @title Segmentation application

segmented_df = split_and_filter(
    df=paper_content.copy(),
    review_key='llm_review_1',
    filter_short_reviews=True,
    do_filter_typos=True,
    consider_only_bullet_points=True,
    exclude_short=False,
    exclude_long=False,
    split_with_llm=False, # Set to False to avoid LLM API calls
    llm_client=None
)

display(segmented_df.head())
display(segmented_df.shape)

Initial number of reviews: 16
Number of reviews after removing  zero-length review 16


100%|██████████| 16/16 [00:00<00:00, 265.39it/s]


Number of reviews with one point: (these reviews will be excluded) 0
Number of all points (for reviews that has more than one point): 219
Number of points after filtering typos and considering only bullet points: 200
filtering reviwes, and only considering the ones with length of one STD away form mean.
Number of reviews before filtering: 16
Number of the review points before filtering: 200
mean: 48.035 std: 68.31115410385043 min: -20.27615410385043 max: 116.34615410385042


100%|██████████| 16/16 [00:00<00:00, 3271.05it/s]

Number of the reviews after filtering: 200


,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes_1,llm_review_1,llm_notes_2,llm_review_2,split_review
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,"[- Simple, scalable method: joint generator/di..."
1,author_3,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: Generating Sequences by Learning to [Se...,# Summary Of The Paper\nThis paper proposes Se...,"Paper: ""Generating Sequences by Learning to [S...",# Summary Of The Paper\nThe paper proposes SEL...,[- Practical and modular: Corrector can be muc...
2,author_4,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,"Paper: ""A Statistical Framework for Personaliz...",# Summary Of The Paper\nThe paper proposes a u...,"Paper: ""A Statistical Framework for Personaliz...",# Summary Of The Paper\nThis paper proposes a ...,[- Strong theory for canonical estimators: Clo...
3,author_5,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,"Paper: ""FROM t-SNE TO UMAP WITH CONTRASTIVE LE...",# Summary Of The Paper\nThis paper provides a ...,"Paper: ""From t-SNE to UMAP with Contrastive Le...",# Summary Of The Paper\nThe paper investigates...,[- Precise identification of UMAP’s implementa...
4,author_6,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,"Paper: ""Reward Learning with Trees: Methods an...",# Summary Of The Paper\nThis paper studies pre...,Title: Reward Learning with Trees: Methods and...,# Summary Of The Paper\nThe paper studies lear...,[- Methodological refinements are sensible and...


(16, 12)

In [32]:
# @title Conversion to long-format

# Each segmented comment becomes a new row
segmented_df_long = segmented_df.explode('split_review')

# Rename the column
segmented_df_long = segmented_df_long.rename(columns={'split_review': 'segmented_comment'})

# Display segmented_df_long
print("Long-format DataFrame head:")
display(segmented_df_long.head())
print("Long-format DataFrame shape:")
display(segmented_df_long.shape)

Long-format DataFrame head:


,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes_1,llm_review_1,llm_notes_2,llm_review_2,segmented_comment
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,"- Simple, scalable method: joint generator/dis..."
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,- Strong empirical gains: consistent improveme...
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,- Thorough analyses: ablations (edges vs featu...
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,- Practical design: compatibility with existin...
0,author_2,gpt-5-mini-2025-08-07,NaN,low,,Faithful,bd301abd5453,Title: DiP-GNN: Discriminative Pre-Training of...,# Summary Of The Paper\nThis paper identifies ...,"Paper: ""DiP-GNN: Discriminative Pre-Training o...",# Summary Of The Paper\nThe paper proposes DiP...,- Theoretical grounding is limited: while the ...


Long-format DataFrame shape:


(200, 12)

In [33]:
# @title Save JSON and Parquet files

# Save the long-format DataFrame to JSONL
segmented_df_long.to_json(RESULTS_JSON_PATH, orient='records', lines=True)
print(f"DataFrame saved to JSONL at: {RESULTS_JSON_PATH}")

# Save the long-format DataFrame to Parquet
segmented_df_long.to_parquet(RESULTS_PATH, index=False)
print(f"DataFrame saved to Parquet at: {RESULTS_PATH}")

DataFrame saved to JSONL at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/llm_reviews_segmented.jsonl
DataFrame saved to Parquet at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews/llm_reviews_segmented.parquet
